# Random Forest

## 1. Load the training data

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import ParameterSampler, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

# Find the data and preprocessing code from either notebook location.
working_dir = Path.cwd()
project_root = next(
    (path for path in (working_dir, *working_dir.parents)
     if (path / 'src' / 'preprocessing_pipeline.py').is_file()
     and (path / 'Dataset' / 'processed').is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Run this notebook from the project root or a subfolder.')

sys.path.insert(0, str(project_root))
from src.preprocessing_pipeline import TARGET, build_preprocessor

RANDOM_STATE = 42
BAD, GOOD = 0, 1
CLASS_NAMES = ['bad', 'good']
data_dir = project_root / 'Dataset' / 'processed'

In [2]:
X_train = pd.read_csv(data_dir / 'X_train.csv')
y_train = pd.read_csv(data_dir / 'y_train.csv')[TARGET]

print(f'Training set has {len(X_train)} rows and {X_train.shape[1]} features.')
display(y_train.value_counts().reindex([BAD, GOOD]).set_axis(CLASS_NAMES).to_frame('count'))

Training set has 700 rows and 21 features.


,count
bad,210
good,490


## 2. Tune the model and threshold

Use the same five stratified folds for every setting. Compare cutoffs for predicting bad credit using the validation probabilities. Rank by mean bad-class F1, then by higher bad-class recall and lower F1 variation across folds.

### Set up the model and settings

In [3]:
baseline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
])
param_grid = {
    'model__n_estimators': [150, 200, 350],
    'model__max_depth': [4, 6, 10, None],
    'model__min_samples_leaf': [1, 3, 6, 12],
    'model__max_features': ['sqrt', 0.3, 0.6],
    'model__class_weight': [None, 'balanced', {BAD: 1.5, GOOD: 1}],
}
sampled_settings = ParameterSampler(param_grid, n_iter=48, random_state=RANDOM_STATE)
all_settings = [{}, *sampled_settings]

### Get validation probabilities

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X_train, y_train))
thresholds = np.arange(20, 61, 5) / 100

# Predict each training row with a model that did not fit that row.
validation_probabilities = []
for params in all_settings:
    candidate = clone(baseline).set_params(**params)
    probabilities = cross_val_predict(
        candidate, X_train, y_train, cv=folds, method='predict_proba',
    )
    validation_probabilities.append(probabilities[:, BAD])

### Compare probability cutoffs

In [5]:
# Reuse the same predictions for each cutoff instead of fitting again.
results = []
for params, bad_probability in zip(all_settings, validation_probabilities):
    for threshold in thresholds:
        predictions = np.where(bad_probability >= threshold, BAD, GOOD)
        fold_f1 = [
            f1_score(
                y_train.iloc[rows], predictions[rows], pos_label=BAD, zero_division=0,
            )
            for _, rows in folds
        ]
        results.append({
            'Parameters': params,
            'Threshold': threshold,
            'Mean F1 (bad)': np.mean(fold_f1),
            'F1 SD': np.std(fold_f1),
            'Recall (bad)': recall_score(y_train, predictions, pos_label=BAD, zero_division=0),
        })

### Pick the best settings

In [6]:
baseline_result = next(
    result for result in results
    if not result['Parameters'] and result['Threshold'] == 0.5
)
ranked_results = pd.DataFrame(results).sort_values(
    ['Mean F1 (bad)', 'Recall (bad)', 'F1 SD'],
    ascending=[False, False, True],
)
best_result = ranked_results.iloc[0]
best_params = best_result['Parameters']
best_threshold = best_result['Threshold']

In [7]:
print(f'Evaluated {len(all_settings)} model settings and {len(thresholds)} thresholds.')
display(pd.DataFrame(
    [baseline_result, best_result.to_dict()], index=['Baseline', 'Selected'],
).drop(columns='Parameters').round(4))

Evaluated 49 model settings and 9 thresholds.


,Threshold,Mean F1 (bad),F1 SD,Recall (bad)
Baseline,0.50,0.4971,0.0427,0.3952
Selected,0.35,0.6152,0.0565,0.8000


In [8]:
selected_params = {
    name.removeprefix('model__'): value for name, value in best_params.items()
}
display(pd.Series(selected_params, name='Selected value', dtype=object).to_frame())

,Selected value
n_estimators,150
min_samples_leaf,3
max_features,sqrt
max_depth,None
class_weight,"{0: 1.5, 1: 1}"


### Fit on the full training set

In [9]:
best_model = clone(baseline).set_params(**best_params).fit(X_train, y_train)

## 3. Evaluate on the held-out test set

Compare the original model and the selected model on the same test rows. The model settings and cutoff come from cross-validation on the training set.

In [10]:
X_test = pd.read_csv(data_dir / 'X_test.csv')
y_test = pd.read_csv(data_dir / 'y_test.csv')[TARGET]

baseline_prediction = baseline.fit(X_train, y_train).predict(X_test)
# Find the probability column for bad credit before applying the cutoff.
bad_column = list(best_model.classes_).index(BAD)
bad_probability = best_model.predict_proba(X_test)[:, bad_column]
y_pred = np.where(bad_probability >= best_threshold, BAD, GOOD)

In [11]:
rows = []
for prediction in (baseline_prediction, y_pred):
    rows.append({
        'Accuracy': accuracy_score(y_test, prediction),
        'Precision (bad)': precision_score(y_test, prediction, pos_label=BAD, zero_division=0),
        'Recall (bad)': recall_score(y_test, prediction, pos_label=BAD, zero_division=0),
        'F1 (bad)': f1_score(y_test, prediction, pos_label=BAD, zero_division=0),
    })
display(pd.DataFrame(rows, index=['Baseline', 'Selected']).round(4))

,Accuracy,Precision (bad),Recall (bad),F1 (bad)
Baseline,0.7967,0.7959,0.4333,0.5612
Selected,0.7000,0.5000,0.7556,0.6018


In [12]:
print('Confusion matrix for the selected model')
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[BAD, GOOD]),
    index=pd.Index(CLASS_NAMES, name='Actual'),
    columns=pd.Index(CLASS_NAMES, name='Predicted'),
))

Confusion matrix for the selected model


Predicted,bad,good
Actual,,
bad,68,22
good,68,142


In [13]:
print(classification_report(
    y_test, y_pred, labels=[BAD, GOOD], target_names=CLASS_NAMES,
    digits=4, zero_division=0,
))

              precision    recall  f1-score   support

         bad     0.5000    0.7556    0.6018        90
        good     0.8659    0.6762    0.7594       210

    accuracy                         0.7000       300
   macro avg     0.6829    0.7159    0.6806       300
weighted avg     0.7561    0.7000    0.7121       300

